# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [1]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("./Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [2]:
import numpy as np
from sklearn.model_selection import KFold, cross_validate

In [8]:
# TODO
# GridSearchCV를 수행하세요.
grid_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 8, 12]
}

grid = GridSearchCV(RandomForestRegressor(random_state=42), grid_params, cv=5, scoring='r2')
start = time.time()
grid.fit(X_train, y_train)
g_end = time.time() - start

i = grid.best_index_
grid_mean, grid_std = grid.cv_results_['mean_test_score'][i], grid.cv_results_['std_test_score'][i]
print(grid.best_params_, grid.best_score_)
print(f'mean={grid_mean}, std={grid_std}, time={g_end}')

{'max_depth': 12, 'n_estimators': 200} 0.8206179847869421
mean=0.8206179847869421, std=0.06442538938888975, time=14.239331007003784


### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  Grid Search는 지정한 조합을 빠짐없이 평가하므로 격자 안의 최적값을 놓치지 않고 재현성이 좋다. 하지만 하이퍼파라미터와 후보 값이 늘면 조합 수가 곱으로 늘어 비용이 급증하고, 격자 사이에 있는 값은 찾을 수 없다.

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [22]:
# TODO
# RandomizedSearchCV를 수행하세요.
random_params = { 
    'n_estimators': randint(100, 301), 
    'max_depth': randint(4, 13), 
}
rand = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), random_params, cv = 5, scoring = 'r2', random_state=42)

start = time.time()
rand.fit(X_train, y_train)
r_end = time.time() - start

i = rand.best_index_
rand_mean, rand_std = rand.cv_results_['mean_test_score'][i], rand.cv_results_['std_test_score'][i]
print(rand.best_params_)
print(f'mean={rand_mean}, std={rand_std}, time={r_end}')

{'max_depth': 11, 'n_estimators': 230}
mean=0.8210296383331108, std=0.0650864730669607, time=7.313228607177734


### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**  탐색할 하이퍼파라미터가 많거나 값의 범위가 넓어서 모든 조합을 다 보기에는 비용이 너무 클 때 유리하다. `n_iter`로 시도 횟수를 직접 정할 수 있어서 시간 예산을 조절하기 쉽고, 같은 횟수로도 Grid 격자에 없는 다양한 값을 시험해볼 수 있다. 이번 실습에서도 Grid와 똑같이 9번만 시도했지만, 격자에 없던 max_depth=11, n_estimators=230 조합을 찾아 Grid(0.8206)보다 조금 높은 CV 평균 0.8210을 얻었다.

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [ ]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.
def objective(trial):
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 300),
        max_depth=trial.suggest_int('max_depth', 4, 12),
        random_state=42,
        n_jobs=-1
    )
    return cross_val_score(model, X_train, y_train, cv = 5, scoring = 'r2').mean()
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=3)
study = optuna.create_study(direction='maximize', sampler=sampler)
start = time.time()
study.optimize(objective, n_trials=9)
o_end = time.time() - start
print(study.best_params, study.best_value, f'time={o_end}')


[I 2026-10-07 14:25:09,333] A new study created in memory with name: no-name-c177ee89-79da-476a-a047-7ac9d9a4ae83
[I 2026-10-07 14:25:10,007] Trial 0 finished with value: 0.8212542415716282 and parameters: {'n_estimators': 175, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 14:25:10,875] Trial 1 finished with value: 0.8184595590761907 and parameters: {'n_estimators': 247, 'max_depth': 9}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 14:25:11,353] Trial 2 finished with value: 0.7990289774524457 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 14:25:11,817] Trial 3 finished with value: 0.8201477777785264 and parameters: {'n_estimators': 109, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 14:25:12,620] Trial 4 finished with value: 0.8205950100254196 and parameters: {'n_estimators': 204, 'max_depth': 12}. Best is trial 0 with value: 0.821254

{'n_estimators': 175, 'max_depth': 12} 0.8212542415716282 time=6.02267599105835


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  Grid와 Random은 이전 결과와 상관없이 후보를 고르지만, Optuna는 이전 trial의 점수를 바탕으로 좋은 결과가 나온 영역을 모델링해 다음 후보를 제안한다. 즉 앞의 시도에서 얻은 정보를 다음 탐색에 활용한다.

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [7]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
opt_scores = cross_val_score(RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1),
                             X_train, y_train, cv=5, scoring="r2")
opt_mean, opt_std = opt_scores.mean(), opt_scores.std()
print(opt_scores.round(4))
print(f"mean={opt_mean:.4f}, std={opt_std:.4f}")

[0.8694 0.6967 0.828  0.8428 0.8694]
mean=0.8213, std=0.0643


---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [25]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.
summary = pd.DataFrame({
    'search_method': ['GridSearchCV', 'RandomizedSearchCV', 'Optuna'],
    'best_params': [grid.best_params_, rand.best_params_, study.best_params],
    'mean_cv_r2': [grid_mean, rand_mean, opt_mean],
    'std_cv_r2': [grid_std, rand_std, opt_std],
    'search_time': [g_end, r_end, o_end]
})
display(summary)
best = summary.loc[summary['mean_cv_r2'].idxmax()]
print(best)

,search_method,best_params,mean_cv_r2,std_cv_r2,search_time
0,GridSearchCV,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,14.239331
1,RandomizedSearchCV,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,7.313229
2,Optuna,"{'n_estimators': 175, 'max_depth': 12}",0.821254,0.064281,6.022676


search_method                                    Optuna
best_params      {'n_estimators': 175, 'max_depth': 12}
mean_cv_r2                                     0.821254
std_cv_r2                                      0.064281
search_time                                    6.022676
Name: 2, dtype: object


### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  무조건 선택할 필요는 없다. 이번 결과에서 세 방법의 CV 평균은 Grid 0.8206, Random 0.8210, Optuna 0.8213으로 차이가 0.001도 나지 않았고, 이는 Fold 간 표준편차(약 0.064~0.065)에 비하면 무시할 수 있는 수준이다. 이렇게 평균이 거의 같을 때는 표준편차와 탐색 비용을 함께 보는 것이 합리적이다. 이번에는 Optuna가 CV 평균이 가장 높으면서 표준편차도 0.0643으로 가장 작았고, 탐색 시간도 약 6초로 Grid보다 짧아서 세 기준 모두에서 가장 좋은 후보로 선택했다. 만약 평균이 아주 조금 높은 모델이 시간을 몇 배 더 쓴다면, 그 차이가 재학습과 운영 비용을 감수할 만큼 의미 있는지 따져보고 선택해야 한다.

In [ ]:
kf = KFold(n_splits=5, shuffle=False)
base = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
tuned = RandomForestRegressor(**best["best_params"], random_state=42, n_jobs=-1)

base_cv = cross_validate(base, X_train, y_train, cv=kf, scoring="r2", return_train_score=True)
tuned_cv = cross_validate(tuned, X_train, y_train, cv=kf, scoring="r2", return_train_score=True)

b_tr, b_val = base_cv["train_score"], base_cv["test_score"]
t_tr, t_val = tuned_cv["train_score"], tuned_cv["test_score"]

fold_comparison = pd.DataFrame({
    "base_train": b_tr,
    "base_val": b_val,
    "tuned_train": t_tr,
    "tuned_val": t_val,
}, index=[1, 2, 3, 4, 5])

tuning_comparison = pd.DataFrame({
    "train_mean": [b_tr.mean(), t_tr.mean()],
    "val_mean":   [b_val.mean(), t_val.mean()],
    "val_std":    [b_val.std(), t_val.std()],
    "gap":        [b_tr.mean() - b_val.mean(), t_tr.mean() - t_val.mean()],
}, index=["base", "tuned"])

r2_diff = t_val.mean() - b_val.mean()
display(fold_comparison, tuning_comparison)
print(f"튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R² = {r2_diff:.4f}")

final_name = "tuned"
final_model = tuned
final_model.fit(X_train, y_train)

test_r2 = final_model.score(X_test, y_test)
final_cv_mean = tuning_comparison.loc[final_name, "val_mean"]
final_cv_std = tuning_comparison.loc[final_name, "val_std"]
gap = test_r2 - final_cv_mean
print(f"Test R²={test_r2:.4f}, Test-CV={gap:.4f}")

os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(final_model, "model_artifacts/best_model.pkl")

meta = {
    "model_name": "RandomForestRegressor",
    "saved_at": datetime.now().isoformat(),
    "search_method": best["search_method"],
    "best_params": best["best_params"],
    "cv_mean": float(final_cv_mean),
    "cv_std": float(final_cv_std),
    "test_r2": float(test_r2),
    "baseline_config": {"n_estimators": 100, "max_depth": None},
    "tuned_config": {"params": best["best_params"], "search_method": best["search_method"]},
    "tuning_comparison": tuning_comparison.to_dict(),
    "cv_r2_diff": float(r2_diff),
    "test_minus_cv": float(gap),
}
with open("model_artifacts/model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=int)
print("저장 완료")

,base_train,base_val,tuned_train,tuned_val
1,0.971728,0.862667,0.971451,0.869414
2,0.975444,0.680790,0.976382,0.696696
3,0.974969,0.834270,0.972136,0.827994
4,0.974586,0.845045,0.972621,0.842770
5,0.971663,0.872282,0.970872,0.869396


,train_mean,val_mean,val_std,gap
base,0.973678,0.819011,0.070370,0.154667
tuned,0.972692,0.821254,0.064281,0.151438


튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R² = 0.0022
Test R²=0.8911, Test-CV=0.0698
저장 완료


### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  탐색 중에 Test를 보고 파라미터나 모델을 바꾸면 Test 정보가 선택 과정에 새어 들어가서 점수가 긍정적으로 부풀려진다. 모든 선택을 Train 내부 CV로 끝낸 뒤 마지막에 한 번만 써야, 처음 보는 데이터에 대한 성능을 공정하게 추정할 수 있다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:** Optuna로 찾은 튜닝 모델(n_estimators=175, max_depth=12)의 CV 평균 R²는 0.8213으로, 기준 모델 0.8190보다 +0.0022 높아졌다. 검증 표준편차는 0.0704에서 0.0643으로 줄었고, Train–검증 차이도 0.1547에서 0.1514로 약간 줄어서 세 지표 모두 개선 방향이다. 특히 가장 낮았던 2번째 Fold가 0.681에서 0.697로 올라간 것이 표준편차가 줄어든 주된 이유이다. 다만 개선 폭이 Fold 간 표준편차(약 0.064)보다 훨씬 작기 때문에 확실한 개선이라기보다는 약간 개선되고 조금 더 안정적이 된 정도로 보는 것이 맞다. 두 모델 모두 2번째 Fold에서만 0.68~0.70으로 크게 떨어지는 것을 보면, 성능 변동은 하이퍼파라미터보다 데이터 자체(이상치 등)의 영향이 더 크다. 또 Train R²가 약 0.97로 검증보다 0.15 정도 높아서 과적합 경향은 튜닝 후에도 남아 있다.

### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:** 최종 모델의 CV 평균은 0.8213이고 hold-out Test R²는 0.8911로, Test가 약 0.07 더 높게 나왔다. Test가 CV보다 낮지 않으므로 Test 기준으로 심한 과적합 신호는 없다. 하지만 Test 샘플이 약 292개로 적고, CV에서 Fold에 따라 0.70~0.87까지 흔들렸던 것을 보면 Test 점수 하나만으로 실제 성능이 0.89라고 단정할 수 없다. 이번 Test 세트에 상대적으로 예측하기 쉬운 집이 많이 들어갔을 가능성도 있다. 또 Train–검증 차이 0.15로 보이는 과적합 경향이 있고, 한 번의 분할 결과만으로는 다른 시기나 지역의 데이터에서도 비슷한 성능이 나올지 보장할 수 없다. Test 결과를 확인한 뒤에는 다시 튜닝하거나 후보를 바꾸지 않았다.


# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?

GridSearchCV는 미리 정한 후보 값들의 모든 조합을 빠짐없이 다 평가하는 방식이고, RandomizedSearchCV는 탐색 범위 안에서 정해진 횟수만큼만 조합을 무작위로 뽑아서 평가하는 방식이다. 그래서 Grid는 격자 안에서는 최적값을 놓치지 않지만 조합이 많아지면 비용이 크게 늘고, Random은 비용을 조절할 수 있는 대신 모든 조합을 보지는 않는다.

2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?

`n_iter`는 무작위로 뽑아서 평가할 하이퍼파라미터 조합의 개수로, 탐색 예산을 직접 정하는 값이다. 이번 실습처럼 `n_iter=9`, `cv=5`이면 9개 조합을 각각 5번씩, 총 45번 학습하게 된다.

3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?

`trial`은 하이퍼파라미터 한 세트를 정해서 objective 함수를 한 번 실행하는 시도이고, 이번 실습에서는 5-Fold CV 평균 R²를 한 번 계산하는 것이 하나의 trial이다. `study`는 이런 trial들을 모두 모아 관리하는 탐색 전체로, 지금까지의 시도 기록과 최적 파라미터(`best_params`), 최고 점수(`best_value`)를 가지고 있다.

4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?

CV 표준편차를 보고 Fold마다 점수가 얼마나 흔들리는지 확인해야 하고, Train 점수와 검증 점수의 차이를 보고 과적합 정도도 확인해야 한다. 또 평균 차이가 아주 작다면 탐색 시간이나 모델 크기 같은 비용도 함께 고려해야 한다. 이번 실습에서도 CV 평균 차이가 0.0001도 안 났기 때문에 표준편차와 탐색 시간을 같이 비교했다.

5. hold-out Test는 언제 사용해야 하나요?

하이퍼파라미터 탐색, 후보 비교, 최종 모델 선택이 모두 끝난 뒤 마지막에 딱 한 번만 사용해야 한다. 중간에 Test를 보고 선택을 바꾸면 Test 정보가 모델 선택에 섞여 들어가서, Test 점수가 더 이상 처음 보는 데이터에 대한 성능이 아니게 되기 때문이다.

6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?

모델을 저장해두면 매번 다시 학습하지 않고 바로 불러와서 예측에 사용할 수 있다. 메타데이터는 이 모델이 어떤 탐색 방법과 파라미터로 만들어졌는지, CV와 Test 점수는 얼마였는지를 기록해두는 것이라서, 나중에 결과를 재현하거나 새 모델과 비교하거나 다른 사람에게 설명할 때 근거로 쓸 수 있다.